# Our method (FVI + per-point decomposition) vs model-free RL baselines

Three external baselines, all facing the identical environment, constraints and evaluation
episodes:

| baseline | family | how it handles `a in {0,1,2}^m` (3^m = 59049 at m=10) |
|---|---|---|
| **BDQ** | value-based | factors `Q` into `m` branches, so `max_a Q` is `m` independent 3-way argmins ([action-branching-agents](https://github.com/atavakol/action-branching-agents), Tavakoli et al. AAAI 2018, MIT) |
| **TRPO** | policy-gradient | **needs no max at all** — critic is `V(s)`, actor is updated on *sampled* actions; a hard KL trust region keeps each step small |

> **PPO was evaluated and dropped.** With the same swap policy and budget it collapses to
> the null policy whenever `c_edit > 0` (taxed random swaps ⇒ negative advantage ⇒ one large
> unconstrained step into the all-keep optimum, with no gradient out). TRPO's KL bound
> prevents exactly that, so TRPO is kept as the policy-gradient representative. Recorded
> here so the omission is not mistaken for cherry-picking; `ppo.py` remains in the repo
> because `trpo.py` reuses its swap-policy helpers.

**The combinatorial action space, three different ways.** Value-based methods must
bootstrap with `max_a Q(s', a)` — exactly the intractable part; BDQ makes it tractable by
factoring. Policy-gradient methods dissolve the max entirely: their critic is a state value
and their actor never argmaxes. What they need instead is a distribution over `3^m` that
can be *sampled from and scored*.

For PPO/TRPO we parameterise that distribution over delete/duplicate **swaps** — the same
coordinates the `balance` constraint is written in:

    r        ~ Categorical(r_logits)                 how many swaps
    del_1..r ~ sequential softmax, no replacement    which points to delete
    dup_1..r ~ sequential softmax, disjoint          which to duplicate

a Plackett-Luce draw, giving feasibility **by construction**, an **exact** log-probability
(so the PPO ratio / TRPO KL are unbiased), and `O(m)` sampling from a space of size `3^m`.

> The obvious alternative — an independent 3-way categorical per point, then project onto
> the feasible set — **fails completely** here, and the reason is worth knowing: with
> `balance=True`, flipping one point to "delete" changes nothing until a matching duplicate
> appears, so the projection erases single-coordinate changes and the policy gradient sits
> on a plateau. PPO collapses to the null policy under every entropy/lr setting. See the
> `ppo.py` docstring.

Our method also has an exact upper bound alongside its practical selector, so **four
learners** are compared:

| learner | action selection | uses the interaction `B`? |
|---|---|---|
| `first-order` | sort on `α` only | no — `B` is dropped |
| `exact` | full BQP over `B` (Gurobi/brute force) | yes — the un-approximated optimum |
| `BDQ` | per-branch argmin + feasibility projection | n/a |
| `TRPO` | swap policy, KL trust region | n/a |

**`low-rank` (rank-`r` dual ascent on `B`) is excluded.** For this problem's `d=5, m=10`,
the interaction matrix `B` is provably rank ≤ 5 and its spectrum is dominated ~3-4x by the
top eigendirection, so low-rank picks nearly-identical actions to first-order even at
`r=1` (>96% agreement, measured) — it adds cost without adding signal in this regime. See
§6's sensitivity markdown for the same finding on the `rank` sweep axis. `exact` is the
more informative upper bound here: it shows how much is left on the table by *any*
`B`-truncation at all, first-order included.

Both method variants are trained **self-consistently** — the tier used at decision time is
also the tier used to collect rollouts — so each one's wall-clock includes the true cost of
its own selector.

---

## Fairness protocol

| axis | control |
|---|---|
| environment | identical `EnvConfig`; identical `theta_star`, `theta_target`, `theta_0` |
| evaluation | identical `eval_seed` + `eval_episodes` → all policies scored on the **same episodes** |
| seeds | same seed list for every learner; mean ± std reported |
| network init | `torch.manual_seed(s)` re-applied before each learner |
| env interactions | `K_iters × N_episodes` episodes each — **matched** |
| replay | BDQ and FVI: off-policy FIFO buffer, same capacity |
| features | PPO/TRPO/BDQ share the **identical** feature set (`theta`, `theta−target`, `t`, per-point `phi_i`, `resid_i`) |
| optimiser | Adam, same `weight_decay` |
| checkpointing | `best_checkpoint=True` for all |
| **constraints** | every policy emits actions from the **same feasible set** (asserted in §5) |
| network capacity | *not* matched — the model-free baselines are far larger (**advantages them**) |
| gradient steps | *not* matched for PPO/TRPO — see below |
| wall-clock | *not* matched at equal `K_iters` — reported separately in §3 |

Asymmetries left in deliberately, all **in the baselines' favour** or inherent:

1. **The model-free baselines have several times more parameters.** Not handicapped — if
   they still lose, the result is robust to capacity.
2. **PPO/TRPO are on-policy**, so each batch is used for a few epochs and discarded; they
   cannot absorb the same gradient-step count as the replay-based learners without
   overfitting their own batch. We therefore match **environment interactions**, the
   standard currency for on-policy vs off-policy comparison, and let gradient steps differ.
3. **`c_edit` is analytic for our method, learned-from-cost for the baselines.** This is not
   a setup flaw; it *is* the hypothesis. Removing it would mean removing the method.

---

**This notebook reads pre-computed results, it does not train anything.** Produce them from the terminal first:

```
./run_seeds.sh --task linear --seeds 10 --mode sort --bdq --trpo   # -> first-order, BDQ, TRPO (+ null/random/greedy)
./run_seeds.sh --task linear --seeds 10 --mode exact                # -> exact (BQP upper bound)
```
(`run_seeds.sh` calls `aggregate.py` automatically once all seeds finish, producing `output/regression_<mode>_results_<timestamp>.json`.) Only §§1-4 below are reproducible from those files -- see the closing note for what isn't and why.

In [ ]:
import glob
import json
import os
import sys
sys.path.insert(0, "src")

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from dataclasses import replace
from config import EnvConfig, MethodConfig, TIER_NAME
from env import Environment
from fvi import train_fvi
from agents import NullAgent, RandomAgent, GreedyAgent, MethodAgent

# NB: do not `import viz` / `import diagnostics` -- they call matplotlib.use("Agg") at
# import time, which disables inline plots. The imports above are only needed for §5's
# live illustration -- everything else in this notebook reads pre-computed output/*.json.

plt.rcParams["figure.dpi"] = 110
plt.rcParams.update({
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9.5,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "0.85",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.6,
})


## Load results

`run.py` trains one selection mode per call (`--mode sort` -> `first-order`, `--mode exact` -> `exact`; `--mode lowrank` also exists but is excluded here, see §0), optionally adding `BDQ`/`TRPO`, and `aggregate.py` folds every seed of one config into a single `output/{task}_{mode}_{tag}_results_{timestamp}.json`.

`sensitivity/*.sh` writes plenty of *other* files into this same `output/` folder, at other configs (different `c_edit`, `m`, ...), so picking "the newest file" or "whichever file has the most seeds" per mode is not safe on its own -- either heuristic can silently pair one mode's headline run with another mode's one-off sensitivity point, badging them as the same comparison. This cell instead fixes an explicit target config up front (`BASELINE`, reused below by §6), and for each mode only accepts a file that (a) has no `sens_axis` tag -- i.e. it came from a plain `run_seeds.sh` call, not a `sensitivity/*.sh` sweep -- and (b) matches `BASELINE` **exactly**. Both checks matter: `c_edit.sh`'s own sweep, for instance, includes a point at exactly `c_edit=0.1` (`BASELINE`'s own value) that otherwise matches `BASELINE` on every other field for the `sort` mode -- `sens_axis`-filtering is what keeps that sensitivity point from being mistaken for the headline run.

Two exact matches can still disagree on *what they trained* -- e.g. a `--bdq --trpo` run and a later plain re-run at the same seed count both match `BASELINE`. Among tied-seed-count candidates, the one recording more learners wins (so a complete run isn't dropped in favour of a newer-but-partial one), with newest path as the final tiebreak. A mode with no match at all is reported and simply left out of the plots below, rather than guessed at.

In [ ]:
TASK = "regression"      # matches run.py's --task for this notebook
OUTDIR = "output"

# The exact config this notebook's headline comparison is about -- mirrors run.py's
# TASK_DEFAULTS["regression"] + the --rank/--k-max argparse defaults. Reused by the
# sensitivity-analysis section (§6) as the point every non-swept axis must match.
BASELINE = {"d": 5, "m": 10, "T": 10, "eta": 0.2, "noise": 0.1, "victim_hidden": 8,
            "init_scale": 0.3, "c_edit": 0.1, "weight_decay": 1e-3, "dual_iters": 20,
            "K_iters": 30, "rank_r": 2, "k_max": None}
MODES = ["sort", "exact"]   # "lowrank" excluded -- see §0's markdown


def cfg_of(run):
    cfg = {**run["env_config"], **run["method_config"]}
    return {f: cfg.get(f) for f in BASELINE}


def matches_baseline(cfg):
    return all(cfg[f] == v for f, v in BASELINE.items())


# Load precisely: for each mode, only accept a file whose recorded config equals BASELINE
# exactly AND whose "sens_axis" tag is unset (i.e. it came from a plain run_seeds.sh call,
# not a sensitivity/*.sh sweep). Both checks matter: c_edit.sh's own sweep, for instance,
# includes a point at exactly c_edit=0.1 (BASELINE's own value) that otherwise matches
# BASELINE on every other field for the "sort" mode -- sens_axis-filtering is what keeps
# that sensitivity point from being mistaken for the headline run.
pattern = os.path.join(OUTDIR, f"{TASK}_*_results_*.json")
all_paths = glob.glob(pattern)
if not all_paths:
    raise SystemExit(
        f"no aggregated results found matching {pattern!r} -- run e.g.\n"
        f"  ./run_seeds.sh --task linear --seeds 10 --mode sort\n"
        f"first (it calls aggregate.py automatically once all seeds finish)")

files, loaded, ambiguous = {}, {}, {}
for mode in MODES:
    candidates = []
    for path in all_paths:
        data = json.load(open(path))
        if data["mode"] != mode or data.get("sens_axis") is not None or not data["runs"]:
            continue
        if matches_baseline(cfg_of(data["runs"][0])):
            candidates.append((path, data))
    if not candidates:
        continue
    # exact-config matches can still differ in *what they trained*: e.g. one --bdq --trpo
    # run and a later plain re-run of the same seed count both match BASELINE exactly.
    # Prefer, in order: more seeds, then more learners recorded (so a complete run beats a
    # partial one at equal seed count), then newest path as a final tiebreak.
    def candidate_key(pd):
        _, d = pd
        return (len(d["runs"]), len(d["runs"][0]["costs"]), pd[0])
    path, data = max(candidates, key=candidate_key)
    if len(candidates) > 1:
        ambiguous[mode] = [os.path.basename(p) for p, _ in candidates]
    files[mode], loaded[mode] = path, data

if not loaded:
    raise SystemExit(
        f"no file in {OUTDIR}/ matches BASELINE={BASELINE} for any of {MODES} -- run e.g.\n"
        f"  ./run_seeds.sh --task linear --seeds 10 --mode sort\n"
        f"first, or edit BASELINE above to match the run you actually produced.")
missing = [m for m in MODES if m not in loaded]

# merge every mode's per-seed runs into one dict keyed by seed. A seed's entry accumulates
# whichever costs/history/train_times/decision_times entries that mode's run.py call
# produced -- e.g. the "sort" run supplies "first-order" + null/random/greedy(+BDQ/TRPO if
# enabled), "exact" supplies "exact" the same way. Where two modes both computed the same
# baseline (e.g. null, or BDQ trained under both runs), the first mode loaded (dict order)
# wins -- they're independent training runs of the same baseline, not literally identical
# numbers.
merged = {}
for mode, data in loaded.items():
    for run in data["runs"]:
        s = run["seed"]
        dst = merged.setdefault(s, {"costs": {}, "history": {}, "train_times": {},
                                     "decision_times": {},
                                     "env_config": run["env_config"],
                                     "method_config": run["method_config"]})
        for key in ("costs", "history", "train_times", "decision_times"):
            for name, val in run.get(key, {}).items():
                dst[key].setdefault(name, val)

SEEDS = sorted(merged)
runs = [merged[s] for s in SEEDS]
ECFG = runs[0]["env_config"]
CFG = runs[0]["method_config"]

PREFERRED_ORDER = ["null", "random", "greedy", "TRPO", "BDQ", "first-order", "exact"]
present = {n for r in runs for n in r["costs"]}
NAMES = [n for n in PREFERRED_ORDER if n in present] + sorted(present - set(PREFERRED_ORDER))
LEARNERS = [n for n in ["first-order", "exact", "BDQ", "TRPO"] if n in present]

# our two selection tiers (first-order, exact) share one marker family ("o") since they're
# the same underlying method, just different selectors -- linestyle tells them apart
# instead (solid vs dotted); baselines each get their own marker+solid line.
COLORS = {"null": "#5b5b5b", "random": "#9c8f8f", "greedy": "#eb6834", "TRPO": "#4a3aa7",
          "BDQ": "#e34948", "first-order": "#2a78d6", "exact": "#12877f"}
MARKERS = {"null": "D", "random": "P", "greedy": "s", "TRPO": "^", "BDQ": "X",
           "first-order": "o", "exact": "o"}
LINESTYLES = {"first-order": "-", "exact": ":"}
_fallback = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for i, n in enumerate(NAMES):
    COLORS.setdefault(n, _fallback[i % len(_fallback)])
    MARKERS.setdefault(n, "o")
    LINESTYLES.setdefault(n, "-")

print("loaded:", {m: os.path.basename(p) for m, p in files.items()})
if missing:
    print(f"no BASELINE-matching file for: {missing} (left out of the plots below)")
if ambiguous:
    for mode, names in ambiguous.items():
        print(f"[{mode}] {len(names)} files match BASELINE -- picked the one with the most "
              f"seeds (ties broken by most learners recorded, then newest): {names}")
print(f"task={TASK}  seeds={SEEDS}  learners={LEARNERS}  names={NAMES}")
print(f"d={ECFG['d']}  m={ECFG['m']}  T={ECFG['T']}  eta={ECFG['eta']}  c_edit={CFG['c_edit']}")


## 1. Headline: mean episode cost

Lower is better. Error bars are ±1 std over seeds.

In [ ]:
cost_mean = {n: float(np.mean([r["costs"][n] for r in runs if n in r["costs"]])) for n in NAMES}
cost_std = {n: float(np.std([r["costs"][n] for r in runs if n in r["costs"]])) for n in NAMES}

# scale figure width and label rotation with the number of bars -- readable at 5 names
# (null/random/greedy/first-order/exact), still readable with BDQ/TRPO added (7)
n = len(NAMES)
fig, ax = plt.subplots(figsize=(max(8, 1.05 * n), 4.6))
ax.bar(range(n), [cost_mean[k] for k in NAMES],
       yerr=[cost_std[k] for k in NAMES], capsize=5, color=[COLORS[k] for k in NAMES])
for i, k in enumerate(NAMES):
    ax.text(i, cost_mean[k] + 0.25, f"{cost_mean[k]:.2f}", ha="center", fontsize=10)
if "null" in cost_mean:
    ax.axhline(cost_mean["null"], color=COLORS["null"], ls="--", lw=1)
ax.set_xticks(range(n))
ax.set_xticklabels(NAMES, rotation=(12 if n <= 6 else 25), fontsize=9,
                    ha=("center" if n <= 6 else "right"))
ax.set_ylabel("mean episode cost (lower = stronger attack)")
ax.set_title(f"Attack strength (d={ECFG['d']}, m={ECFG['m']}, T={ECFG['T']}, "
             f"c_edit={CFG['c_edit']}, {len(SEEDS)} seeds)")
plt.tight_layout()
plt.show()

if "null" in cost_mean:
    nul = cost_mean["null"]
    for k in NAMES:
        if k != "null":
            print(f"{k:12} vs null: {(nul - cost_mean[k]) / nul * 100:5.1f}%")
tiers_present = [t for t in ("first-order", "exact") if t in cost_mean]
if len(tiers_present) > 1:
    base = cost_mean[tiers_present[0]]
    for t in tiers_present[1:]:
        d = base - cost_mean[t]
        print(f"\n{tiers_present[0]} vs {t}: {d:+.3f} ({d / base * 100:+.2f}%)")


## 2. Learning curves — training iterations

`history[name]` is the held-out mean episode cost after every training iteration (`train_fvi`/`train_bdq`/`train_trpo` called with `return_history=True`, evaluated on the same fixed `eval_seed` every iteration -- a clean curve). Learners can have different curve lengths (TRPO's `K_iters` defaults to `max(K_iters, 50)`, longer than FVI's), so each is plotted against its own iteration count rather than a shared axis.

In [ ]:
def curve_stats(name):
    H = np.array([r["history"][name] for r in runs if name in r["history"]])   # (seeds, K)
    return H.mean(0), H.std(0)

fig, ax = plt.subplots(figsize=(9.5, 5.2))
for name in LEARNERS:
    mu, sd = curve_stats(name)
    it = np.arange(1, len(mu) + 1)
    ax.plot(it, mu, marker=MARKERS[name], linestyle=LINESTYLES[name], color=COLORS[name],
            lw=2, label=name, markevery=max(1, len(it) // 8))
    # lighter shading with more curves on the axes keeps overlaps from turning into a smear
    ax.fill_between(it, mu - sd, mu + sd, color=COLORS[name], alpha=0.12 if len(LEARNERS) > 3 else 0.15)
if "null" in cost_mean:
    ax.axhline(cost_mean["null"], color=COLORS["null"], ls="--", lw=1.5,
               label=f"null ({cost_mean['null']:.2f})")
if "greedy" in cost_mean:
    ax.axhline(cost_mean["greedy"], color=COLORS["greedy"], ls="-.", lw=1.5,
               label=f"greedy ({cost_mean['greedy']:.2f})")
ax.set_xlabel("training iteration")
ax.set_ylabel("held-out mean episode cost")
ax.set_title("Learning curves (shaded = ±1 std over seeds)")
ax.legend(fontsize=9, ncol=(2 if len(LEARNERS) + 2 > 5 else 1))
sec = ax.secondary_xaxis("top", functions=(lambda k: k * CFG["N_episodes"] * ECFG["T"],
                                            lambda n: n / (CFG["N_episodes"] * ECFG["T"])))
sec.set_xlabel("environment transitions consumed (approx.: N_episodes × T per iteration)")
plt.show()


## 3. Learning curves — wall-clock

Matched iterations is not matched compute. `train_times[name] / len(history[name])` gives each learner's mean seconds-per-iteration; re-plotting against cumulative wall-clock answers *"given a fixed compute budget, which is ahead?"*

In [ ]:
per_it = {n: float(np.mean([r["train_times"][n] / len(r["history"][n])
                             for r in runs if n in r["train_times"] and n in r["history"]]))
          for n in LEARNERS}

fig, ax = plt.subplots(figsize=(9.5, 5.2))
for name in LEARNERS:
    mu, sd = curve_stats(name)
    it = np.arange(1, len(mu) + 1)
    x = it * per_it[name]
    ax.plot(x, mu, marker=MARKERS[name], linestyle=LINESTYLES[name], color=COLORS[name], lw=2,
            label=f"{name} ({per_it[name]:.1f} s/iter)", markevery=max(1, len(it) // 8))
    ax.fill_between(x, mu - sd, mu + sd, color=COLORS[name], alpha=0.12 if len(LEARNERS) > 3 else 0.15)
if "greedy" in cost_mean:
    ax.axhline(cost_mean["greedy"], color=COLORS["greedy"], ls="-.", lw=1.5, label="greedy")
if "null" in cost_mean:
    ax.axhline(cost_mean["null"], color=COLORS["null"], ls="--", lw=1.5, label="null")
ax.set_xscale("log")
ax.set_xlabel("cumulative training wall-clock (s, log scale)")
ax.set_ylabel("held-out mean episode cost")
ax.set_title("Same curves against compute spent")
ax.legend(fontsize=8, ncol=(2 if len(LEARNERS) + 2 > 5 else 1))
plt.show()

for name in LEARNERS:
    mu, _ = curve_stats(name)
    print(f"  {name:12} final={mu[-1]:.3f}  reached in {per_it[name] * len(mu):6.1f}s "
          f"({per_it[name]:.2f} s/iter x {len(mu)} iters)")


## 4. Training time & decision latency

Mean wall-clock to train each learner once (one seed) -- a one-off cost -- and mean
wall-clock per single `agent.act()` call at decision time (timed directly around that
call in `fvi.rollout`, see `run.py`'s `evaluate()`). Decision latency is recorded for
every evaluated policy, including the untrained baselines (`null`/`random`/`greedy`),
since it's an inference-time cost independent of whether a policy was trained at all.

In [ ]:
train_t = {n: float(np.mean([r["train_times"][n] for r in runs if n in r["train_times"]]))
           for n in LEARNERS}
train_t = dict(sorted(train_t.items(), key=lambda kv: kv[1]))   # ascending: fastest first

fig, ax = plt.subplots(figsize=(max(7, 1.3 * len(train_t)), 4.3))
ax.bar(range(len(train_t)), list(train_t.values()), color=[COLORS[k] for k in train_t])
ax.set_xticks(range(len(train_t)))
ax.set_xticklabels(list(train_t), rotation=12, fontsize=9)
for i, v in enumerate(train_t.values()):
    ax.text(i, v + max(train_t.values()) * 0.02, f"{v:.0f}s", ha="center", fontsize=9)
ax.set_ylabel("seconds")
ax.set_title("Mean training time per seed")
plt.tight_layout()
plt.show()

if "first-order" in train_t:
    for n in LEARNERS:
        if n != "first-order":
            print(f"{n}/first-order training time = {train_t[n] / train_t['first-order']:.1f}x")

# decision latency covers every evaluated policy (NAMES), not just the trained tiers --
# null/random/greedy have no training time but still "decide" every step, so their
# inference cost is just as meaningful a comparison point here. Guard against older
# results.json files (pre-dating this field) contributing nothing for a name -- rather
# than a NaN bar, that name is just left out and reported.
dec_names = [n for n in NAMES if any(n in r.get("decision_times", {}) for r in runs)]
if not dec_names:
    print("no decision_times recorded in the loaded runs -- re-run with the current "
          "run.py to get this (older results.json files predate it)")
else:
    dec_t = {n: float(np.mean([r["decision_times"][n] for r in runs if n in r.get("decision_times", {})])) * 1000
              for n in dec_names}   # seconds -> ms
    dec_t = dict(sorted(dec_t.items(), key=lambda kv: kv[1]))   # ascending: fastest first
    missing_dec = [n for n in NAMES if n not in dec_names]
    if missing_dec:
        print(f"no decision_times for: {missing_dec} (older results.json files predate this field)")

    fig, ax = plt.subplots(figsize=(max(7, 1.3 * len(dec_t)), 4.3))
    ax.bar(range(len(dec_t)), list(dec_t.values()), color=[COLORS[k] for k in dec_t])
    ax.set_xticks(range(len(dec_t)))
    ax.set_xticklabels(list(dec_t), rotation=12, fontsize=9)
    for i, v in enumerate(dec_t.values()):
        ax.text(i, v + max(dec_t.values()) * 0.02, f"{v:.3f}", ha="center", fontsize=9)
    ax.set_ylabel("milliseconds / decision")
    ax.set_title("Mean decision latency (single agent.act() call)")
    plt.tight_layout()
    plt.show()

    if "first-order" in dec_t:
        for n in dec_names:
            if n != "first-order" and dec_t[n] > 0:
                print(f"{n}/first-order decision latency = {dec_t[n] / dec_t['first-order']:.1f}x")


### Configuration for the live illustration below

§§1-4 above and §6 below are entirely CLI-driven -- everything comes from
`output/regression_*.json`. §5's learned-line plot is the one exception: it needs a
live trained value net and live rollouts to draw, which no `results.json` summary can
provide (same limitation noted in the closing section). This cell defines the
`EnvConfig`/`MethodConfig` objects that feed it, matching the same `BASELINE` used above
(`run.py`'s `TASK_DEFAULTS["regression"]`) -- nothing is trained yet.

In [ ]:
ECFG_LIVE = EnvConfig(d=5, m=10, T=10, eta=0.2)
CFG_LIVE  = MethodConfig(select_mode="sort", c_edit=0.1, weight_decay=1e-3,
                         K_iters=30, rank_r=2)
EVAL_SEED_LIVE = 999


## 5. The learned regression line under each attack policy (d = 1)

In 1-D the victim's entire hypothesis is a line, so the attack is directly visible. A
separate `d=1` run with **every policy on the identical data stream** -- null, random,
greedy, TRPO, BDQ, and ours. Same idea as `experiment_nn.ipynb`'s §3, adapted for a
linear (not MLP) victim.

**Reading the panels.** Marker shape is the attacker's action (● kept, ✕ deleted,
▲ duplicated). Four lines appear in every panel:

| line | meaning |
|---|---|
| grey dotted | the teacher `y = theta* . x` that generated the data |
| **green dashed** | `f(.; theta~)` -- the line the attacker's *target weights* encode |
| blue thin | clean SGD (null) -- where the victim lands with no attack |
| **black thick** | `f(.; theta_T)` -- where *this* policy drove the victim |

A successful attack pulls the black line off the blue one and toward the green one. Two
numbers per panel: `||theta_T - theta~||` is the quantity the cost actually minimises,
and RMSE to the target line measures the same thing in *function* space.
`experiment_nn.ipynb`'s version of this section reports both because weight-space
symmetry can make two different `theta` encode nearly the same MLP function -- a linear
model at `d=1` has no such symmetry, so here the two numbers are exactly proportional
(RMSE = `||theta_T - theta~|| x RMS(x_grid)`, a fixed constant); both are kept anyway for
consistency with the other notebook.

In [ ]:
INCLUDE_MODEL_FREE_LINE = True     # BDQ ~3.5 min, TRPO ~1 min; False = skip both

E1, C1 = replace(ECFG_LIVE, d=1), replace(CFG_LIVE, c_edit=0.01)
torch.manual_seed(0)
env1 = Environment(E1, C1)
GAP_SCALE = 4.0   # widen ||theta_target|| so the (small, d=1) per-point edit benefit
env1.theta_target = env1.theta_target * GAP_SCALE  # clears BDQ's Q-learning noise floor
print(f"d=1 regression victim: theta in R^{env1.d}")

V1 = train_fvi(env1, C1, seed=0, verbose=False, best_checkpoint=True,
               eval_episodes=100, eval_seed=EVAL_SEED_LIVE)

policies = {
    "null (clean SGD)": NullAgent(),
    "random": RandomAgent(torch.Generator().manual_seed(2), C1.balance, C1.k_max),
    "greedy": GreedyAgent(env1.victim, env1.theta_target, C1, mode="sort"),
}
if INCLUDE_MODEL_FREE_LINE:
    from bdq import train_bdq, BDQAgent
    from trpo import train_trpo, TRPOAgent
    torch.manual_seed(0)
    Q1 = train_bdq(env1, C1, seed=0, verbose=False, best_checkpoint=True,
                   eval_episodes=100, eval_seed=EVAL_SEED_LIVE)
    torch.manual_seed(0)
    P1 = train_trpo(env1, replace(C1, K_iters=50), seed=0, verbose=False,
                    best_checkpoint=True, eval_episodes=100, eval_seed=EVAL_SEED_LIVE)
    policies["TRPO"] = TRPOAgent(P1, env1, C1, torch.Generator().manual_seed(1))
    policies["BDQ"] = BDQAgent(Q1, env1, C1, torch.Generator().manual_seed(1))
policies[TIER_NAME[C1.select_mode]] = MethodAgent(V1, env1.victim, env1.theta_target, C1,
                                        torch.Generator().manual_seed(1))

rng = torch.Generator().manual_seed(5)
batches = [env1.stream.sample_batch(env1.m, rng) for _ in range(env1.T)]   # shared stream


def run_on(agent):
    th = env1.theta_0.clone()
    xs, ys, acts = [], [], []
    for t, z in enumerate(batches):
        a = agent.act(th, z, t)
        xs.append(z[0].reshape(-1)); ys.append(z[1]); acts.append(a)
        th = env1.victim.update(th, z, a).detach()
    return th, torch.cat(xs).numpy(), torch.cat(ys).numpy(), torch.cat(acts).numpy()


X_all = torch.cat([z[0].reshape(-1) for z in batches]).numpy()
xs_grid = torch.linspace(float(X_all.min()), float(X_all.max()), 300)[:, None]
f = lambda th: (xs_grid @ th).numpy()
f_star = (xs_grid.reshape(-1) * env1.stream.theta_star).numpy()   # linear teacher, y = theta*.x
f_targ = f(env1.theta_target)
f_null = f(run_on(NullAgent())[0])

ncols = 3
nrows = (len(policies) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.5 * nrows),
                         sharex=True, sharey=True, squeeze=False)
rows = []
for ax, (name, ag) in zip(axes.flat, policies.items()):
    th, X, Y, A = run_on(ag)
    keep, dele, dup = A == 1, A == 0, A == 2
    ax.scatter(X[keep], Y[keep], s=13, c="0.55", alpha=.45, zorder=2)
    ax.scatter(X[dele], Y[dele], s=75, c="#b3202c", marker="x", linewidths=2.3, zorder=4)
    ax.scatter(X[dup], Y[dup], s=62, c="#1f9c4e", marker="^",
               edgecolors="k", linewidths=.8, zorder=4)
    ax.plot(xs_grid.numpy(), f_star, ":", color="0.35", lw=1.8, zorder=3)
    ax.plot(xs_grid.numpy(), f_targ, "--", color="#00b050", lw=2.4, zorder=5)
    ax.plot(xs_grid.numpy(), f_null, "-", color="tab:blue", lw=1.6, alpha=.75, zorder=5)
    ax.plot(xs_grid.numpy(), f(th), "-", color="k", lw=3.0, zorder=6)
    gap = float((th - env1.theta_target).norm())
    fdist = float(np.sqrt(((f(th) - f_targ) ** 2).mean()))
    rows.append((name, gap, fdist, int(dele.sum()), int(dup.sum())))
    ax.set_title(f"{name}\n||theta_T - theta~|| = {gap:.2f}   RMSE to target line = {fdist:.2f}\n"
                 f"{int(dele.sum())} deleted / {int(dup.sum())} duplicated", fontsize=9.5)
    ax.set_xlabel("x"); ax.set_ylabel("y / f(x)")
for ax in list(axes.flat)[len(policies):]:
    ax.axis("off")

from matplotlib.lines import Line2D
fig.legend(handles=[
    Line2D([0], [0], color="0.35", ls=":", lw=1.8, label="teacher  y = theta* . x"),
    Line2D([0], [0], color="#00b050", ls="--", lw=2.4, label="target  f(.; theta~)"),
    Line2D([0], [0], color="tab:blue", lw=1.6, label="clean SGD (null)"),
    Line2D([0], [0], color="k", lw=3.0, label="this policy's f(.; theta_T)"),
    Line2D([0], [0], marker="o", color="0.55", ls="None", label="kept"),
    Line2D([0], [0], marker="x", color="#b3202c", ls="None", label="deleted"),
    Line2D([0], [0], marker="^", color="#1f9c4e", mec="k", ls="None", label="duplicated")],
    loc="lower center", ncol=7, fontsize=9)
fig.suptitle("Regression victim (d=1): the learned line under each attack policy -- "
             "identical data stream", fontsize=13)
fig.tight_layout(rect=(0, 0.07, 1, 0.95)); plt.show()

print(f"{'policy':20}{'||theta_T-theta~||':>20}{'RMSE to target line':>21}{'del':>6}{'dup':>6}")
for r in rows:
    print(f"{r[0]:20}{r[1]:>20.3f}{r[2]:>21.3f}{r[3]:>6d}{r[4]:>6d}")
print("\nSingle episode, single seed -- illustrative; the quantitative comparison is §1.")


## 6. Sensitivity analysis

Data for this section comes from `sensitivity/*.sh` (`c_edit.sh`, `eta.sh`, `m.sh`,
`T.sh`, `d.sh`, `k_max.sh` -- `rank.sh` is excluded by default, see below), each of which
retrains at every value of one axis and writes an
`output/{task}_{mode}_{tag}_results_{timestamp}.json` per value via the usual
`run_seeds.sh` + `aggregate.py` path -- so every axis's sweep points sit in the same
`output/` folder as the headline runs above.

Membership is read from each file's own `sens_axis` field -- a tag `run.py --sens-axis`
writes verbatim, set automatically by each `sensitivity/*.sh` script -- rather than
inferred from config content. An earlier version of this cell tried to infer membership
by checking that every field *other* than the swept one matched the task's baseline;
that broke for `m`/`k_max`, whose own scripts deliberately move `c_edit` alongside their
axis (so `c_edit` had to be exempted from the check), because the exemption couldn't
distinguish "m.sh's own c_edit variation" from an unrelated `c_edit.sh` sweep point that
happened to land at the same `m` value -- silently pooling the two into one badly
corrupted point. Trusting the tag each script writes itself rules that out. Any axis you
haven't (re-)run since this fix landed is skipped with a message -- run the matching
script under `sensitivity/` to fill it in.

**`rank` is dropped from the default sweep.** For this task's `d=5, m=10`, the
interaction matrix `B` is provably rank <= `d` = 5 (it's built as `Phi @ (.) @ Phi.T`
with `Phi` an `m x d` matrix), and its spectrum is dominated ~3-4x by the top
eigendirection -- so `r=1..5` pick nearly-identical actions (>96% agreement even at
`r=1`) and the sweep is flat well within seed noise. See `sensitivity/rank.sh`'s header
for how to re-add it (a larger `--d`, or the `mlp` task, breaks the ceiling).

**Each plot now has up to three panels**: mean episode cost (as before), mean training
time per seed, and mean decision latency (a single `agent.act()` call) -- all against
the same x-axis. Training time only applies to the trained tiers; decision latency
applies to every evaluated policy including the untrained baselines. A panel with no
data for the current axis (e.g. decision latency on data collected before that field was
added to `run.py`) is simply omitted rather than shown empty.

In [ ]:
# field: which env_config/method_config key is this axis's x-value.  mode: which tier's
# output file to read.  "rank" is intentionally absent -- see the markdown above.
AXIS_SPECS = {
    "c_edit": dict(mode="sort",    field="c_edit", xlabel="edit price  c_edit"),
    "eta":    dict(mode="sort",    field="eta",    xlabel="victim learning rate  eta"),
    "m":      dict(mode="sort",    field="m",      xlabel="batch size  m"),
    "T":      dict(mode="sort",    field="T",      xlabel="horizon  T"),
    "d":      dict(mode="sort",    field="d",      xlabel="problem dimension  d"),
    "k_max":  dict(mode="sort",    field="k_max",  xlabel="edit budget  k_max"),
}


def load_sensitivity(axis):
    """{"cost": {value: {learner: [costs]}}, "train_time": {...}, "decision_time": {...}}

    Trusts each file's own "sens_axis" tag (written by run.py --sens-axis, set
    automatically by the matching sensitivity/*.sh script) instead of inferring axis
    membership from config content -- see the markdown above for why that inference
    was unsafe for m/k_max specifically.

    k_max=None (unconstrained) never belongs on the k_max axis (that's the *absence* of
    a budget, not a swept value) -- k_max.sh's own sweep values are always concrete
    ints, so this can only trigger on a mis-tagged file; excluded defensively below.

    train_time/decision_time are pooled the same way as cost (one raw value per seed,
    from that run's own "train_times"/"decision_times" dicts) -- a name simply won't
    appear in a metric's points if no matching run recorded it (e.g. null has no
    train_time; older results.json files predate decision_time entirely).
    """
    spec = AXIS_SPECS[axis]
    pattern = os.path.join(OUTDIR, f"{TASK}_*_results_*.json")
    metrics = {"cost": {}, "train_time": {}, "decision_time": {}}
    sources = {"cost": "costs", "train_time": "train_times", "decision_time": "decision_times"}
    for path in glob.glob(pattern):
        data = json.load(open(path))
        if data["mode"] != spec["mode"] or data.get("sens_axis") != axis:
            continue
        for run in data["runs"]:
            cfg = {**run["env_config"], **run["method_config"]}
            v = cfg[spec["field"]]
            if v is None:
                continue
            for metric, src in sources.items():
                for name, val in run.get(src, {}).items():
                    metrics[metric].setdefault(v, {}).setdefault(name, []).append(val)
    return metrics


def _errorbar_panel(ax, points, xlabel, ylabel, scale=1.0):
    """One axis's worth of mean+-std error bars, one line per learner present."""
    values = sorted(points)
    names = [n for n in PREFERRED_ORDER if any(n in points[v] for v in values)]
    for name in names:
        xs = [v for v in values if name in points[v]]
        mu = [np.mean(points[v][name]) * scale for v in xs]
        sd = [np.std(points[v][name]) * scale for v in xs]
        ax.errorbar(xs, mu, yerr=sd, marker=MARKERS.get(name, "o"), color=COLORS.get(name),
                    lw=2, capsize=4, label=name)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return names


def plot_sensitivity(axis):
    spec = AXIS_SPECS[axis]
    metrics = load_sensitivity(axis)
    if not metrics["cost"]:
        print(f"[{axis}] no matching runs in {OUTDIR}/ yet -- run ./sensitivity/{axis}.sh")
        return

    # (metric key, subplot ylabel, unit scale) -- ms reads better than seconds for a
    # single decision; a panel with no data for this axis (e.g. decision_time on runs
    # collected before that field existed) is dropped rather than shown empty.
    panel_specs = [("cost", "mean episode cost", 1.0),
                   ("train_time", "train time (s)", 1.0),
                   ("decision_time", "decision latency (ms)", 1000.0)]
    panel_specs = [p for p in panel_specs if metrics[p[0]]]

    fig, axes = plt.subplots(1, len(panel_specs), figsize=(6.6 * len(panel_specs), 4.3), squeeze=False)
    legend_handles = None
    for ax, (metric, ylabel, scale) in zip(axes[0], panel_specs):
        _errorbar_panel(ax, metrics[metric], spec["xlabel"], ylabel, scale)
        if legend_handles is None:
            legend_handles = ax.get_legend_handles_labels()
    n_seeds = max(len(next(iter(metrics["cost"][v].values()))) for v in metrics["cost"])
    fig.suptitle(f"Sensitivity: {spec['xlabel']}  (up to {n_seeds} seed(s)/point)")
    axes[0][0].legend(*legend_handles, fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
for axis in AXIS_SPECS:
    plot_sensitivity(axis)


## Not reproducible from `results.json`

`run.py` deliberately saves only costs, per-iteration held-out cost (`history`), per-learner training time (`train_times`), and per-policy decision latency (`decision_times`) -- no model weights, no environment, no rollouts. That rules out rebuilding, from this file alone:

- **behaviour** (edits/step, distance to target) -- needs a live trained agent + environment to roll out further. Decision latency itself **is** covered now (§4) -- it doesn't need a rollout, just timing `agent.act()` once per evaluation step, which `run.py` already does.
- **the interaction-term diagnostic** (relative size of `B`, rank agreement) -- needs the trained `V̂` and per-state `(α, B)` traces; `run.py --diagnostics` still produces these three figures as PNGs (not JSON) if you need them
- **equal-budget comparison and scaling in `m`'s action-space-size framing (§6 in the original live notebook)** -- these need more than cost alone (edits/step, wall-clock vs `3^m`) at each point, not just a retrained cost number

The `c_edit`/`eta`/`m`/`T`/`d`/`k_max` sensitivity sweeps **are** now covered -- §6 above reads them from `sensitivity/*.sh`'s output (`rank` is deliberately excluded there, see its markdown). The **learned regression line** (§5) is also now covered, but not from `results.json` -- it's a live, single-seed illustration trained directly in this notebook, the one exception to the "everything from `output/`" rule; see §5's own intro for why. Any of the still-missing items could be added the same way `history`/`train_times`/`decision_times` were: extend what `run.py` computes and writes into `results.json`, without ever saving a checkpoint.